In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import MusicDBIO
from utils import MusicDBPermDir
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from lib import metalarchives
mio   = metalarchives.MusicDBIO(verbose=False, mkDirs=False)
apiio = metalarchives.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant MetalArchives DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/MetalArchives


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Master Search:       {0}".format(len(localArtists.get())))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

MetalArchives Search Results
   Local Master Search:       4622
   Global Master Search:      20402
   Global Master Search Data: 0
   Search Artists:            20402
   Errors:                    90
   Known Summary IDs:         4597


# Search For New Artists

In [ ]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

In [ ]:
useStarterData = True
useMasterData  = False

if useStarterData is True:
    starterData = io.get("starter.p")
    artistNames = Series({v["Ref"].split("/")[-1]: v["Name"] for k,v in starterData.items()})
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import MusicDBIO
    pdbio = MusicDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    

#   Artist Names To Get:  36777
#   Artist Names To Get:  31070
#   Artist Names To Get:  22044


## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "5:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        apiio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    apiio.sleep(3.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
    masterArtists.save(data=masterArtistsDict)
    masterArtistsData.save(data=masterArtistsDataDict)
    if len(searchedForErrors) > 0:
        errors.save(data=searchedForErrors)

## Save Results

In [ ]:
df = masterArtistsData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = metalarchives.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = searchDF.append(Series(df))
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    artists = {}
    for artistName,artistResults in searchDF.iteritems():
        if artistResults is not None:
            for item in artistResults:
                artists[item['id']] = item['name']
    print("Found {0} Unique Artists".format(len(artists)))
    s = Series(artists)
    print("  ==> {0} Old Artists".format(len(s[s.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(s[~s.index.isin(knownArtists.index)])))
    print("Saving Data")
    metalarchives.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

## Find Artists To Download

In [7]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

MetalArchives Search Results
   Available IDs:      54178
   Known Artist IDs:   4622
   Artist IDs To Get:  49556


## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Starting Process [Getting MetalArchives Artist Data]    ==> Time Is 2022-03-20 16:00:36
========================= termTime(day=today,time=9:00pm) =========================
   ====> Terminate Time Set To 2022-03-20 21:00:00 <====
   ====> Will Terminate Process 4 Hours and 59 Minutes From Now
Searching For Glass Skeleton Death March (bands/_/3540362144)   True
Searching For Original Chaotic Addiction (bands/_/3540267573)   True
Searching For Ides of Gemini (bands/_/3540437335)               True
Searching For Doubled Over (bands/_/3540295797)                 True
Searching For Unlight Dawn (bands/_/3540285036)                 True
5/?        : Process [Getting MetalArchives Artist Data] Has Run For 1 Minute and 2 Seconds.
Saving 4627 MetalArchives Searched For Artist (Info) IDs
Searching For Shit No More (bands/_/3540293056)                 True
Searching For Down Under (bands/_/46351)                        True
Searching For Beyond the Pain (bands/_/3540320693)              True
Searc

Searching For Invocation (bands/_/6943)                         True
Searching For Ritual (bands/_/3540444714)                       True
Searching For As Death Lingers (bands/_/3540364032)             True
Searching For Летаргия (bands/_/3540304043)                     True
Searching For Counterforce (bands/_/3540484123)                 True
55/?       : Process [Getting MetalArchives Artist Data] Has Run For 11 Minutes and 28 Seconds.
Saving 4677 MetalArchives Searched For Artist (Info) IDs
Searching For Jesus Cröst (bands/_/3540446202)                  True
Searching For Gravelord (bands/_/3540479846)                    True
Searching For Lethal Gram (bands/_/3540444719)                  True
Searching For Regredior (bands/_/25323)                         True
Searching For Landvaettir (bands/_/12536)                       True
60/?       : Process [Getting MetalArchives Artist Data] Has Run For 12 Minutes and 31 Seconds.
Saving 4682 MetalArchives Searched For Artist (Info) IDs
Sear

Searching For Empire Night (bands/_/84637)                      True
Searching For Abyssal Storm (bands/_/3540408131)                True
Searching For Metal Macbeth (bands/_/3540293635)                True
Searching For Chaos Legion (bands/_/83957)                      True
105/?      : Process [Getting MetalArchives Artist Data] Has Run For 21 Minutes and 53 Seconds.
Saving 4727 MetalArchives Searched For Artist (Info) IDs
Searching For Fire Action (bands/_/3540450339)                  True
Searching For Putrefacción (bands/_/3540423394)                 True
Searching For Bells of Doom (bands/_/3958)                      True
Searching For Lifeless Vomit (bands/_/42096)                    True
Searching For Reflection of Desires (bands/_/3540488595)        True
110/?      : Process [Getting MetalArchives Artist Data] Has Run For 22 Minutes and 54 Seconds.
Saving 4732 MetalArchives Searched For Artist (Info) IDs
Searching For Art Like Destruction (bands/_/107740)             True
Sear

Searching For Genesis of Pain (bands/_/100205)                  True
Searching For Operation Winter Mist (bands/_/5239)              True
155/?      : Process [Getting MetalArchives Artist Data] Has Run For 32 Minutes and 8 Seconds.
Saving 4777 MetalArchives Searched For Artist (Info) IDs
Searching For Altar of Woe (bands/_/3540469745)                 True
Searching For Vanaheim (bands/_/3540425863)                     True
Searching For Dark Pariah (bands/_/3540276064)                  True
Searching For Shadow of the Colossus (bands/_/3540325248)       True
Searching For Paindeath (bands/_/68467)                         True
160/?      : Process [Getting MetalArchives Artist Data] Has Run For 33 Minutes and 9 Seconds.
Saving 4782 MetalArchives Searched For Artist (Info) IDs
Searching For Ravenous (bands/_/3540441636)                     True
Searching For Last Empire (bands/_/17004)                       True
Searching For Carnal Tomb (bands/_/3540387350)                  True
Search

Searching For Dark Plague (bands/_/3540382300)                  True
Searching For Nightmare (bands/_/3540313339)                    True
Searching For PostHumanBigBang (bands/_/3540333800)             True
Searching For Machinations of Fate (bands/_/3540362008)         True
Searching For Nausea (bands/_/3540292546)                       Error ==> Nausea
Searching For Borean Dusk (bands/_/3540313959)                  True
210/?      : Process [Getting MetalArchives Artist Data] Has Run For 43 Minutes and 40 Seconds.
Saving 4832 MetalArchives Searched For Artist (Info) IDs
Searching For Bloodred History (bands/_/3540319737)             True
Searching For Necronautical (bands/_/3540381694)                True
Searching For Nightstorm (bands/_/59151)                        True
Searching For Holier than Thou? (bands/_/96413)                 True
Searching For Kingdom Faust (bands/_/3540402020)                True
215/?      : Process [Getting MetalArchives Artist Data] Has Run For 44 Minu

...

Searching For Gregory's Xit (bands/_/3540376965)                True
Searching For Scarlet Dew (bands/_/114990)                      True
Searching For Black Goat Desecrator (bands/_/3540410877)        True
Searching For Quidam 666 (bands/_/104877)                       True
Searching For Zath (bands/_/3540350645)                         True
465/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 37 Minutes.
Saving 5087 MetalArchives Searched For Artist (Info) IDs
Searching For Pale Existence (bands/_/10233)                    True
Searching For Ancient Sickness (bands/_/3540301309)             True
Searching For Tyranny at Large (bands/_/3540496041)             True
Searching For Dripping Orifice (bands/_/3540299533)             True
Searching For Guerra Total (bands/_/3540256808)                 True
470/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 38 Minutes.
Saving 5092 MetalArchives Searched For Artist (Info) IDs
Searching Fo

Searching For Inoculation (bands/_/13463)                       True
Searching For Boss (bands/_/10171)                              True
Searching For Darken My Grief (bands/_/41850)                   True
Searching For Angst (bands/_/3540341549)                        True
Searching For The Growling Winds (bands/_/3540424400)           True
515/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 48 Minutes.
Saving 5137 MetalArchives Searched For Artist (Info) IDs
Searching For Jealous Wild (bands/_/3540438569)                 True
Searching For Midnight Hero (bands/_/129088)                    True
Searching For Goremouth (bands/_/3540499075)                    True
Searching For Northcrown (bands/_/15065)                        True
Searching For StarGazer (bands/_/2942)                          True
520/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 49 Minutes.
Saving 5142 MetalArchives Searched For Artist (Info) IDs
Searching Fo

Searching For Seven Witches (bands/_/476)                       True
Searching For Metalion (bands/_/3540356387)                     True
565/?      : Process [Getting MetalArchives Artist Data] Has Run For 1 Hour and 59 Minutes.
Saving 5187 MetalArchives Searched For Artist (Info) IDs
Searching For Monarch (bands/_/102735)                          True
Searching For Ossian (bands/_/74896)                            True
Searching For Thy Serpent (bands/_/717)                         True
Searching For Violent Omen (bands/_/3540311061)                 True
Searching For Slaughter the Prophets (bands/_/3540353642)       True
570/?      : Process [Getting MetalArchives Artist Data] Has Run For 2 Hours and 24 Seconds.
Saving 5192 MetalArchives Searched For Artist (Info) IDs
Searching For Transe Metal Machine (bands/_/3540357435)         True
Searching For Dread (bands/_/44193)                             True
Searching For Quark 7 (bands/_/43883)                           True
Searching F

Searching For Tortura Silenciosa (bands/_/3540495414)           True
Searching For Neon Cross (bands/_/3812)                         True
Searching For Sojourner (bands/_/53611)                         True
Searching For America Gomorrah (bands/_/69411)                  True
Searching For Beyond Boundaries (bands/_/3540465333)            True
620/?      : Process [Getting MetalArchives Artist Data] Has Run For 2 Hours and 11 Minutes.
Saving 5242 MetalArchives Searched For Artist (Info) IDs
Searching For Skull of Monthu (bands/_/3540352026)              True
Searching For Worsen (bands/_/3540376714)                       True
Searching For Bottled (bands/_/68027)                           True
Searching For Neuronal Eradication (bands/_/3540427085)         True
Searching For Accid Reign (bands/_/84856)                       True
625/?      : Process [Getting MetalArchives Artist Data] Has Run For 2 Hours and 12 Minutes.
Saving 5247 MetalArchives Searched For Artist (Info) IDs
Searching 

Searching For Aggressor (bands/_/31859)                         True
Searching For Paranormal Slaughter (bands/_/3540407298)         True
670/?      : Process [Getting MetalArchives Artist Data] Has Run For 2 Hours and 21 Minutes.
Saving 5292 MetalArchives Searched For Artist (Info) IDs
Searching For Ira Inferos (bands/_/3540496839)                  True
Searching For Акелдама (bands/_/3540417567)                     True
Searching For Odium (bands/_/34251)                             True
Searching For Ninjas for Hire (bands/_/3540359427)              True
Searching For Coven of the Wretched (bands/_/3540492554)        True
675/?      : Process [Getting MetalArchives Artist Data] Has Run For 2 Hours and 22 Minutes.
Saving 5297 MetalArchives Searched For Artist (Info) IDs
Searching For Vorace (bands/_/57194)                            True
Searching For Abhorrent (bands/_/276)                           True
Searching For Algol (bands/_/2764)                              True
Searching 

In [ ]:
ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
mio.data.getRawArtistInfoData(mio.mv.get(3540473060), 3540473060)

In [ ]:
localArtistsDict

In [ ]:
localArtists.save(data=localArtistsDict)


In [ ]:
tracks
disc_count